# Notebook 4 — Baseline Decision Tree (instructions only)

**Research Project:** Lightweight Machine Learning-Based Cybersecurity Threat Detection for Resource-Constrained Hardware
**Author:** Anush Jindal

**There is no code in this notebook on purpose.** Each step tells you *what* to do, *what to use*, and *how to know it worked*. You write the code in the empty cells. If you get stuck, read the linked documentation, check `PIPELINE_CHEATSHEET.ipynb`, and ask.

## The goal of this stage

Train the simplest possible model (a Decision Tree with **no tuning**) on each of your four clue lists (A, B, C, D). Record how well it scores and how big and slow it is. This is the **baseline**: the reference point we try to shrink later.

## Three rules

1. **Learn from `X_train` only.** `X_test` is the exam. Use it to score, never to choose clues or settings.
2. **Don't tune anything yet.** No `max_depth`, no pruning. That is a later stage.
3. **Write down the numbers you actually get.** Never a guessed number.

## Steps

1. Setup
2. Load your data and your four clue lists
3. Test your own keep/remove claims
4. Train on clue list A and score it
5. Compare A, B, C, D
6. Shortcut check
7. Save your results
8. Read your numbers honestly

## Step 1 — Setup

**Goal:** load the tools and tell the notebook where your files are.

**What to do:**
- Import `pandas`, `numpy`, `matplotlib.pyplot`, `seaborn` and `joblib`.
- From scikit-learn, import `DecisionTreeClassifier` (from `sklearn.tree`) and these from `sklearn.metrics`: `accuracy_score`, `precision_recall_fscore_support`, `classification_report`, `confusion_matrix`.
- Also import the standard modules `os`, `time`, `json` and `datetime`.
- In Colab, mount your Google Drive and save the path of the folder with your data files (the same folder you used in Notebook 3) in a variable.

**Docs:** [DecisionTreeClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.tree.DecisionTreeClassifier.html) · [Model evaluation](https://scikit-learn.org/stable/modules/model_evaluation.html)

**You're done when:** the cell runs with no error and printing your folder path shows the right place.

## Step 2 — Load your data and your four clue lists

**Goal:** get the study pile, the exam pile and your four lists of column names into the notebook.

**What to do:** load these five files from your data folder:

| File | What it is |
|---|---|
| `X_train_engineered.csv` | the clues for the study pile |
| `X_test_engineered.csv` | the clues for the exam pile |
| `y_train.csv` | the answer key for the study pile |
| `y_test.csv` | the answer key for the exam pile |
| `feature_sets.json` | your lists A, B, C, D |

**Tips:**
- The two `y` files each have one column. You want a plain list of labels, not a one-column table. Look up what pandas calls turning a one-column table into a single column (hint: `squeeze`).
- Use the `json` module to read the clue lists.

**Checks to run (print or assert them):**
- The shapes match your notes from Notebook 2.
- The four lists have **57, 39, 15 and 6** names.
- Train and test have the same columns in the same order.
- Every name in every list really exists in `X_train`.
- Count the labels in `y_train`. Make sure the exact text `BENIGN` is one of them.

**You're done when:** every check passes. If one fails, stop and fix it before moving on.

## Step 3 — Test your own keep/remove claims

**Goal:** your feature table says some columns are *identical* or *always zero*. That came from reading about the dataset, not from your data. Check it. **If a claim is false, the data wins:** fix your table and your `A_drop` list in Notebook 3 and re-save.

**Part A — claimed identical.** For each pair, test whether the two columns are the same on every row (numpy has a comparison that allows for tiny decimal differences: look up `allclose`).

| Dropped column | Compare with |
|---|---|
| Avg Fwd Segment Size | Fwd Packet Length Mean |
| Avg Bwd Segment Size | Bwd Packet Length Mean |
| Subflow Fwd Packets | Total Fwd Packets |
| Subflow Fwd Bytes | Total Length of Fwd Packets |
| Subflow Bwd Packets | Total Backward Packets |
| Subflow Bwd Bytes | Total Length of Bwd Packets |
| Fwd Header Length.1 | Fwd Header Length |
| ECE Flag Count | RST Flag Count |

**Part B — claimed same ordering.** Compute the Spearman correlation (pandas `corr` has a `method` option) for *Packet Length Variance* vs *Packet Length Std*, and *Average Packet Size* vs *Packet Length Mean*. A value of 1.0 means a tree gains nothing from having both.

**Part C — claimed always zero (or nearly).** For each of these columns, find how many different values it has (`nunique`) and what share of rows are not zero:
Fwd PSH Flags, Bwd PSH Flags, Fwd URG Flags, Bwd URG Flags, CWE Flag Count, and the six Bulk columns (Fwd and Bwd: Avg Bytes/Bulk, Avg Packets/Bulk, Avg Bulk Rate).

**How to read it:**
- Identical = `True` means your reason was right.
- `False` means your reason was wrong. Decide again and write what you decided.
- One distinct value means the column never changes, so it carries no information.

**You're done when:** you have written one line in `experiments/experiment_notes.md` for each claim that turned out wrong (or "all claims confirmed").

## Step 4 — Train on clue list A and score it

**Goal:** train one tree on list A and measure both *how good* and *how expensive* it is.

**What to do:**
1. Create a `DecisionTreeClassifier` with `random_state=42` and **no other settings**.
2. Time the training. Use `time.perf_counter()` before and after `fit`, using only the columns in list A from `X_train`, and `y_train`.
3. Time the prediction on `X_test` (again only list A's columns).
4. Measure:
   - accuracy
   - macro precision, macro recall and macro F1 (`precision_recall_fscore_support` with `average="macro"` and `zero_division=0`)
   - **attack vs normal:** call a row an "attack" if its label is not `BENIGN`. Count **caught** (real attack, guessed attack), **missed** (real attack, guessed normal) and **false alarms** (really normal, guessed attack). Then attack recall = caught ÷ (caught + missed).
   - tree size: number of nodes (`model.tree_.node_count`) and depth (`model.get_depth()`)
   - saved model size: save the model with `joblib.dump`, then ask the operating system for the file size (`os.path.getsize`) and convert to KB
5. Print a `classification_report` (per-class scores).
6. Draw the confusion matrix as a heatmap. Divide each row by its total so each row shows *the share of that true class*. Rare attacks then stay visible.

**Tip:** put steps 1 to 4 inside **one function** that takes a name and a list of columns and returns your results. You will reuse it in Steps 5 and 6.

**Words to know**
- **macro F1:** the average across all classes, counting rare attacks as much as common ones. Harsher than accuracy.
- **attack recall:** of all real attacks, how many did we catch? Missing one is the worst mistake.

**Docs:** [classification_report](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html) · [confusion_matrix](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.confusion_matrix.html) · [precision_recall_fscore_support](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.precision_recall_fscore_support.html) · [joblib](https://joblib.readthedocs.io/)

**Do not tune the tree.** Leave every setting at its default except `random_state`.

**You're done when:** you have a results dictionary for list A, a printed report, and a heatmap. In the heatmap, find one row where the diagonal is not near 1.0 and note which class it is confused with.

## Step 5 — Compare A, B, C, D

**Goal:** answer the first piece of the research question: *how much do we lose when we use fewer clues?*

**What to do:**
- Run your function from Step 4 on lists B, C and D.
- Put all four results into one table (`pandas.DataFrame`), sorted from most clues to fewest.
- Draw two small graphs: macro F1 against number of clues, and saved model size against number of clues.

**Before you look:** write in your notes whether you expect the score to fall a lot or a little as the clues shrink. Then compare.

**Do not decide anything from one run.** Just read it.

**You're done when:** you have the table and both graphs.

## Step 6 — Shortcut check

**The worry (it is in your own feature table):** CICIDS2017 attacks were run from a lab against fixed ports and fixed machines. A tree might score well just by memorising *"this port or this window size means attack"* instead of learning how attacks behave. A real network would not hand us that shortcut.

**What to do:** run your function again on these clue lists, and add them to your results:

| Run name | Clues |
|---|---|
| A_no_port | list A without `Destination Port` |
| A_no_port_no_window | list A without `Destination Port`, `Init_Win_bytes_forward` and `Init_Win_bytes_backward` |
| C_no_port | list C without `Destination Port` |
| D_no_port | list D without `Destination Port` |

**How to read it:** if the score barely moves, the tree is learning behaviour. If it collapses, it was leaning on a shortcut, and that is an important finding. This is *not* choosing clues with the exam. We are asking a question about the model, not tuning it. Still, write down what you did and why.

**You're done when:** you have four more rows in your results table and one sentence in your notes about what changed.

## Step 7 — Save your results

**Goal:** write your numbers in the same layout as `experiments/results.csv` in the repo.

**The columns, in this order:**
`experiment_id, date, stage, dataset_file, num_samples, num_features, features_used, max_depth, num_nodes, accuracy, precision, recall, f1, model_size_kb, training_time_s, inference_ms, ram_mb, notes`

**How to fill them:**
- `experiment_id`: for example `S5-A`, `S5-B`, `S5-A_no_port`
- `stage`: `5-baseline`
- `dataset_file`: `Wednesday-workingHours`
- `num_samples`: train rows + test rows
- `features_used`: the run name (A, B, C, D, A_no_port, ...)
- `max_depth`: the **actual** depth of the tree you got (you did not set it)
- `precision`, `recall`, `f1`: the **macro** averages
- `inference_ms`: milliseconds **per 1000 rows**, and say so in `notes`
- `ram_mb`: **leave empty.** We have not measured memory. Never guess it.
- `notes`: say it is an untuned tree, and include attack recall, missed attacks and false alarms

**Then:**
1. Save a CSV in your Drive folder.
2. Copy the rows into `experiments/results.csv` in the repo.
3. Add a dated entry to `experiments/experiment_notes.md`: what you saw and what surprised you.
4. Push, including `feature_sets.json` (put it in `experiments/`). Never push the data CSVs.

**You're done when:** the repo shows your new rows on GitHub.

## Step 8 — Read your numbers honestly

Check each of these before you write any conclusion:

- **A very high score?** Don't celebrate yet. You split one day of traffic at random, so near-identical rows from the same attack burst can land in both piles. A high score here does not prove it would work on a different day or a different network.
- **Rare attack classes.** A class with only a handful of test rows (Heartbleed has very few) can swing wildly. Report it, but don't build a claim on it.
- **Compare attack recall with accuracy.** Missing real attacks matters more than a few false alarms.
- **Did the shortcut check change things?** If yes, say so plainly. That is a finding, not a failure.
- **Never write a number you did not see on screen.**

## What comes next (do not start yet)

The next stage shrinks the tree on purpose and watches what happens to these same numbers. A good question to bring: *at which point does the score start to fall faster than the size does?*